In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, URL

load_dotenv(".env", override=True)

url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM patients", engine)
print(df.shape)

(297, 14)


In [2]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training set:", X_train.shape, "Disease rate:", round(y_train.mean(), 3))
print("Test set:", X_test.shape, "Disease rate:", round(y_test.mean(), 3))

Training set: (237, 13) Disease rate: 0.46
Test set: (60, 13) Disease rate: 0.467


In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

dummy_accuracy = accuracy_score(y_test, dummy.predict(X_test))
print("Lazy guesser accuracy:", round(dummy_accuracy, 3))

Lazy guesser accuracy: 0.533


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

categorical_cols = ["cp", "restecg", "slope", "thal"]
numeric_cols = ["age", "sex", "trestbps", "chol", "fbs", "thalach", "exang", "oldpeak", "ca"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

log_reg = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])

log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["No disease", "Disease"]))

[[28  4]
 [ 5 23]]
              precision    recall  f1-score   support

  No disease       0.85      0.88      0.86        32
     Disease       0.85      0.82      0.84        28

    accuracy                           0.85        60
   macro avg       0.85      0.85      0.85        60
weighted avg       0.85      0.85      0.85        60



In [5]:
from sklearn.base import clone
from xgboost import XGBClassifier

xgb = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", XGBClassifier(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
    )),
])

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb, target_names=["No disease", "Disease"]))

[[29  3]
 [ 4 24]]
              precision    recall  f1-score   support

  No disease       0.88      0.91      0.89        32
     Disease       0.89      0.86      0.87        28

    accuracy                           0.88        60
   macro avg       0.88      0.88      0.88        60
weighted avg       0.88      0.88      0.88        60



In [6]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in [("Logistic Regression", log_reg), ("XGBoost", xgb)]:
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=["accuracy", "recall", "f1"])
    print(name)
    for metric in ["accuracy", "recall", "f1"]:
        values = scores[f"test_{metric}"]
        print(f"  {metric}: {values.mean():.3f} (+/- {values.std():.3f})")
    print()

Logistic Regression
  accuracy: 0.814 (+/- 0.083)
  recall: 0.751 (+/- 0.110)
  f1: 0.787 (+/- 0.090)

XGBoost
  accuracy: 0.810 (+/- 0.056)
  recall: 0.760 (+/- 0.102)
  f1: 0.784 (+/- 0.068)



In [7]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import recall_score, precision_score

train_probs = cross_val_predict(xgb, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
y_true = y_train.to_numpy()

rows = []
for threshold in [0.5, 0.45, 0.4, 0.35, 0.3]:
    preds = (train_probs >= threshold).astype(int)
    rows.append({
        "threshold": threshold,
        "recall": round(recall_score(y_true, preds), 3),
        "precision": round(precision_score(y_true, preds), 3),
        "missed_sick": int(((preds == 0) & (y_true == 1)).sum()),
        "false_alarms": int(((preds == 1) & (y_true == 0)).sum()),
    })

print(pd.DataFrame(rows))

   threshold  recall  precision  missed_sick  false_alarms
0       0.50   0.761      0.814           26            19
1       0.45   0.780      0.780           24            24
2       0.40   0.798      0.777           22            25
3       0.35   0.817      0.774           20            26
4       0.30   0.835      0.740           18            32


In [8]:
from sklearn.metrics import roc_auc_score

THRESHOLD = 0.35

test_probs = xgb.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= THRESHOLD).astype(int)

print(confusion_matrix(y_test, test_preds))
print(classification_report(y_test, test_preds, target_names=["No disease", "Disease"]))
print("ROC AUC:", round(roc_auc_score(y_test, test_probs), 3))

[[29  3]
 [ 3 25]]
              precision    recall  f1-score   support

  No disease       0.91      0.91      0.91        32
     Disease       0.89      0.89      0.89        28

    accuracy                           0.90        60
   macro avg       0.90      0.90      0.90        60
weighted avg       0.90      0.90      0.90        60

ROC AUC: 0.915


In [9]:
import joblib

os.makedirs("models", exist_ok=True)

joblib.dump(
    {"model": xgb, "threshold": THRESHOLD},
    "models/heart_disease_model.joblib"
)

print("Model saved!")

Model saved!
